In [1]:
import json
import re
import pandas as pd
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.preprocessing import MultiLabelBinarizer

# ---------------- CONFIG ----------------
PRED_FILE = "/content/naija_predictions.json"

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# ---------------- 1. DEFINE CLINICAL LABELS ----------------
# Standard CheXpert-style keyword mapping (Simplified for Python)
# This maps findings in the text to specific pathology labels.

CHEXPERT_PATTERNS = {
    "Cardiomegaly": [
        "cardiomegaly", "enlarged heart", "heart size is enlarged",
        "cardiac silhouette is enlarged", "multichamber configuration",
        "left ventricular configuration", "ctr >", "ctr is 5", "ctr is 6", "ctr is 7"
    ],
    "Edema": [
        "edema", "oedema", "vascular congestion", "cephalization",
        "upper lobe diversion", "engorgement", "vascular prominence", "fluid overload"
    ],
    "Consolidation": [
        "consolidation", "air bronchogram", "dense opacity", "alveolar opacity",
        "air-space opacity", "pneumonia"
    ],
    "Pleural Effusion": [
        "effusion", "costophrenic angle", "costophrenic sulcus", "blunting",
        "meniscus", "fluid level", "hydro-pneumothorax"
    ],
    "Pneumothorax": [
        "pneumothorax", "collapsed lung", "pleural line", "air in pleural space"
    ],
    "TB/Opacities": [ # Adding this as it is common in Naija datasets
        "tuberculosis", "ptb", "reticulonodular", "cavitation", "fibrosis",
        "opacity", "opacities", "hazy", "haziness", "infiltrate"
    ]
}

In [ ]:
def get_labels(text):
    """
    Scans text for keywords and returns a list of detected pathologies.
    Returns ['No Finding'] if no pathologies are found.
    """
    text = text.lower()
    detected = []

    # 1. Check for specific pathologies
    found_any = False
    for pathology, keywords in CHEXPERT_PATTERNS.items():
        # Check if any keyword exists in text
        if any(k in text for k in keywords):
            # Exclude negations (Simple rule-based negation handling)
            # e.g., "No evidence of effusion" should not trigger "Effusion"
            # We check if the match is preceded by "no " or "negative for "
            is_negated = False
            for k in keywords:
                if k in text:
                    # Find index
                    idx = text.find(k)
                    # Look at 20 chars before
                    context = text[max(0, idx-20):idx]
                    if "no " in context or "free" in context or "clear" in context or "without" in context:
                        is_negated = True
                        break

            if not is_negated:
                detected.append(pathology)
                found_any = True

    # 2. Check for "No Finding" explicitly or implicitly
    # If text explicitly says "normal" and we haven't found a specific disease yet
    if not found_any:
        if "normal" in text or "unremarkable" in text or "clear" in text:
            detected.append("No Finding")
        else:
            # If unclear, we often default to No Finding or leave empty.
            # For this eval, we'll label 'No Finding' if nothing else triggered.
            detected.append("No Finding")

    return detected

In [ ]:
# ---------------- 2. LOAD & LABEL DATA ----------------
print(f"Loading {PRED_FILE}...")
with open(PRED_FILE, 'r') as f:
    data = json.load(f)

y_true_labels = []
y_pred_labels = []

print("Extracting clinical labels from reports...")

for item in data:
    # Get text
    gt_text = item["ground_truth"]
    pred_text = item["prediction"]

    # Extract Labels
    gt_labels = get_labels(gt_text)
    pred_labels = get_labels(pred_text)

    y_true_labels.append(gt_labels)
    y_pred_labels.append(pred_labels)

Loading /content/naija_predictions.json...


FileNotFoundError: [Errno 2] No such file or directory: '/content/naija_predictions.json'